# ISIC 2019 — EfficientNet-B0 Architecture

[PyTorch Data Tutorial](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html) 

ISIC 2019 data is downloaded and placed under `DATA_ROOT` below.

In [42]:
#!pip install timm -q

In [43]:
#to fix the error from build_model(); Error displaying widget: model not found
#from tqdm.notebook import tqdm

In [44]:
#IGNORE---TESTING; pre-download weights and cache EfficientNet-B0 weights for later use in build_model()
#import timm
#timm.create_model("efficientnet_b0", pretrained=True)
#print("Weights cached!")

In [25]:
import os, json, time, warnings
import wandb
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn

import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR #baseline
from torchvision import transforms #for iamge preprocessing, e.g. resize, augmentation etc

from sklearn.model_selection import GroupShuffleSplit #for lesion_id split later

warnings.filterwarnings("ignore")
print("Imports OK")

Imports OK


In [26]:
import subprocess, zipfile

# PATHS 
DATA_ROOT = "/scratch/umw7eg/isic2019"
CKPT_DIR  = "/scratch/umw7eg/isic2019/checkpoints"
os.makedirs(DATA_ROOT, exist_ok=True)

BASE_URL = "https://isic-archive.s3.amazonaws.com/challenges/2019"

In [27]:
# Download CSVs & zips 
DOWNLOADS = [
    (f"{BASE_URL}/ISIC_2019_Training_GroundTruth.csv", f"{DATA_ROOT}/train_gt.csv"),
    (f"{BASE_URL}/ISIC_2019_Training_Metadata.csv",    f"{DATA_ROOT}/train_meta.csv"),
    (f"{BASE_URL}/ISIC_2019_Test_GroundTruth.csv",     f"{DATA_ROOT}/test_gt.csv"),
    
    (f"{BASE_URL}/ISIC_2019_Training_Input.zip",       f"{DATA_ROOT}/train_imgs.zip"),
    (f"{BASE_URL}/ISIC_2019_Test_Input.zip",           f"{DATA_ROOT}/test_imgs.zip"),
]

for url, dest in DOWNLOADS:
    print(f"Downloading {os.path.basename(dest)} ...")
    subprocess.run(
        ["curl", "--location", "--progress-bar",
         "--retry", "3", "--retry-delay", "5",
         "--output", dest, url],
        check=True,
    )

######################################################################## 100.0%
                                                                           1.3%

######################################################################## 100.0%
######################################################################## 100.0%


######################################################################## 100.0%


######################################################################## 100.0%


In [28]:
# Unzip
for zip_path, extract_dir in [
    (f"{DATA_ROOT}/train_imgs.zip", DATA_ROOT),
    (f"{DATA_ROOT}/test_imgs.zip",  DATA_ROOT),
]:
    print(f"Extracting {os.path.basename(zip_path)} ...")
    
    # Sanity check
    size_mb = os.path.getsize(zip_path) / (1024**2)
    print(f"  File size: {size_mb:.1f} MB")
    if size_mb < 1:
        raise RuntimeError(f"File too small — download likely failed: {zip_path}")
    
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    
    os.remove(zip_path)
    print(f"  Done & removed {os.path.basename(zip_path)}")

Extracting train_imgs.zip ...
  File size: 9318.9 MB
  Done & removed train_imgs.zip
Extracting test_imgs.zip ...
  File size: 3646.1 MB
  Done & removed test_imgs.zip


In [29]:
# Path 
TRAIN_DIR  = os.path.join(DATA_ROOT, "ISIC_2019_Training_Input")
TEST_DIR   = os.path.join(DATA_ROOT, "ISIC_2019_Test_Input")
TRAIN_CSV  = os.path.join(DATA_ROOT, "train_gt.csv")
TEST_CSV   = os.path.join(DATA_ROOT, "test_gt.csv")
TRAIN_META = os.path.join(DATA_ROOT, "train_meta.csv")

print("Done! Data ready.")

Done! Data ready.


## Config for ablation studies

In [30]:
#os.makedirs(CKPT_DIR, exist_ok=True)

Config = {
    # model
    "architecture": "efficientnet_b0",  # "resnet50" | "mobilenetv3_small" | efficientnet_b0
    "pretrained":   True,              # False = train from scratch ablation
    "freeze_bb":    True,             # True = head-only training ablation, so frozen backbone
    "loss_fn":      "weighted_ce",      # "ce" | "focal"
    "augmentation": "standard",           # "none" | "geometric" | "color" | "standard"
    "img_size":     224,
    "batch_size":   32,
    "epochs":       30,
    "lr":           1e-4,
    "val_split":    0.20,
    "patience":     10,                #stop epoch if no improvement for 10
    "seed":         42,                #set_seed for reproducibility 
    "classes":      ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"],
}
Config["num_classes"] = len(Config["classes"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Config: {Config}")

Device: cuda
Config: {'architecture': 'efficientnet_b0', 'pretrained': True, 'freeze_bb': True, 'loss_fn': 'weighted_ce', 'augmentation': 'standard', 'img_size': 224, 'batch_size': 32, 'epochs': 30, 'lr': 0.0001, 'val_split': 0.2, 'patience': 10, 'seed': 42, 'classes': ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC'], 'num_classes': 8}


In [31]:
import random

def set_seed(seed=Config["seed"]):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
print(f"Seed set to {Config['seed']}")

Seed set to 42


In [32]:
#Dataset 

class ISICDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(f"{self.img_dir}/{row['image']}.jpg").convert("RGB")
        
        if self.transform:
            img = self.transform(img)
        label = int(row["label"])
        return img, label

def get_transforms(split):
    aug = Config["augmentation"]
    
    if split in ("val", "test"):
        return transforms.Compose([
            transforms.Resize((Config["img_size"], Config["img_size"])),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
    
    # Train transforms vary by augmentation setting in the Config code cell
    if aug == "none":
        aug_tfms = []
    elif aug == "geometric":
        aug_tfms = [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(degrees=15),
        ]
    elif aug == "color":
        aug_tfms = [
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        ]
    elif aug == "standard":
        aug_tfms = [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.RandomRotation(degrees=15),
        ]
    
    return transforms.Compose([
        transforms.Resize((Config["img_size"], Config["img_size"])),
        *aug_tfms,
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

In [33]:
import timm #using timm library as per Prof feedback
#from huggingface_hub import login
#model = timm.create_model("mobilenetv3_small_100", pretrained=True)

def build_model(num_classes=Config["num_classes"]):    
    model = timm.create_model("efficientnet_b0",
                              pretrained=Config["pretrained"],
                              num_classes=num_classes)
    return model

model = build_model().to(device)

print(type(model))
print("Model loaded OK")

<class 'timm.models.efficientnet.EfficientNet'>
Model loaded OK


## Training loop

In [44]:
import torch.nn.functional as F
from sklearn.metrics import (
    balanced_accuracy_score, confusion_matrix, roc_auc_score,
    precision_score, recall_score, f1_score, 
    ConfusionMatrixDisplay,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [45]:
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            probs   = F.softmax(outputs, dim=1)
            preds   = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)
    num_cls    = Config["num_classes"]
    cls_range  = list(range(num_cls))

    # ── Core metrics ──────────────────────────────────────────────────────────
    acc  = (all_preds == all_labels).mean()
    bacc = balanced_accuracy_score(all_labels, all_preds)

    # ── Confusion matrix ──────────────────────────────────────────────────────
    cm = confusion_matrix(all_labels, all_preds, labels=cls_range)

    # ── Per-class: sensitivity, specificity, precision, recall, F1 ───────────
    sensitivity, specificity, f1_per, precision_per, recall_per = [], [], [], [], []

    for i in range(num_cls):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp

        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = sens                                         # recall == sensitivity
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

        sensitivity.append(sens)
        specificity.append(spec)
        precision_per.append(prec)
        recall_per.append(rec)
        f1_per.append(f1)

    sensitivity   = np.array(sensitivity)
    specificity   = np.array(specificity)
    precision_per = np.array(precision_per)
    recall_per    = np.array(recall_per)
    f1_per        = np.array(f1_per)

    # ── Aggregate: macro & weighted precision / recall / F1 ──────────────────
    macro_precision = precision_score(all_labels, all_preds, average="macro",    zero_division=0)
    macro_recall    = recall_score   (all_labels, all_preds, average="macro",    zero_division=0)
    macro_f1        = f1_score       (all_labels, all_preds, average="macro",    zero_division=0)
    wtd_precision   = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    wtd_recall      = recall_score   (all_labels, all_preds, average="weighted", zero_division=0)
    wtd_f1          = f1_score       (all_labels, all_preds, average="weighted", zero_division=0)

    # ── AUC per class ─────────────────────────────────────────────────────────
    auc_per_class = {}
    for i, cls in enumerate(Config["classes"]):
        try:
            auc_per_class[cls] = roc_auc_score(
                (all_labels == i).astype(int), all_probs[:, i]
            )
        except ValueError:
            auc_per_class[cls] = float("nan")

    aggregates = {
        "macro_precision": macro_precision,
        "macro_recall":    macro_recall,
        "macro_f1":        macro_f1,
        "wtd_precision":   wtd_precision,
        "wtd_recall":      wtd_recall,
        "wtd_f1":          wtd_f1,
    }

    return (
        acc, bacc,
        sensitivity, specificity,
        precision_per, recall_per, f1_per,
        auc_per_class,
        cm,
        aggregates,
    )


def plot_confusion_matrix(cm, class_names, title="Confusion Matrix", save_path=None):
    """Print a text table then plot a row-normalised heatmap of the confusion matrix."""
    # ── printed version ───────────────────────────────────────────────────────
    print(f"\n{title}")
    header = "      " + "  ".join(f"{c:>5}" for c in class_names)
    print(header)
    for i, row_cls in enumerate(class_names):
        row_str = "  ".join(f"{v:5d}" for v in cm[i])
        print(f"  {row_cls:>4}  {row_str}")

    # ── plotted version (row-normalised → recall) ─────────────────────────────
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(
        cm_norm,
        annot=False,
        fmt="",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        linewidths=0.4,
        linecolor="#cccccc",
        vmin=0, vmax=1,
        ax=ax,
    )

    # Dual annotation: raw count (bold) + recall % underneath
    for row in range(len(class_names)):
        for col in range(len(class_names)):
            pct = cm_norm[row, col]
            color = "white" if pct > 0.55 else "black"
            ax.text(col + 0.5, row + 0.38, f"{cm[row, col]}",
                    ha="center", va="center", fontsize=8, fontweight="bold", color=color)
            ax.text(col + 0.5, row + 0.65, f"{pct:.1%}",
                    ha="center", va="center", fontsize=7, color=color)

    ax.set_xlabel("Predicted label", fontsize=11)
    ax.set_ylabel("True label",      fontsize=11)
    ax.set_title(title, fontsize=10, pad=12)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Confusion matrix saved → {save_path}")
    plt.show()

In [46]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

# Convert label column
train_df["label"] = train_df[Config["classes"]].values.argmax(axis=1)
test_df["label"]  = test_df[Config["classes"]].values.argmax(axis=1)

# Train/val split stratified by label...to be improved later by adding patient-level split with lesion_id(GroupShuffleSplit)
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    train_df,
    test_size=Config["val_split"],
    stratify=train_df["label"],
    random_state=Config["seed"],
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 20264 | Val: 5067 | Test: 8238


In [47]:
#show class imabalnce in frequency table
counts        = train_df["label"].value_counts().sort_index()
CLASS_FREQ    = (counts / counts.sum()).to_dict()

print("Class frequencies:")

for name, df_ in [("TRAIN", train_df), ("VAL", val_df)]:
    labels = df_[Config["classes"]].values.argmax(axis=1)
    cnts   = np.bincount(labels, minlength=len(Config["classes"]))

    print(f"\n{name} ({len(df_)} images):")
    for i, c in enumerate(Config["classes"]):
        print(f"  {c:6s}: {cnts[i]:5d}  ({100*cnts[i]/len(df_):4.1f}%)")

Class frequencies:

TRAIN (20264 images):
  MEL   :  3618  (17.9%)
  NV    : 10300  (50.8%)
  BCC   :  2658  (13.1%)
  AK    :   694  ( 3.4%)
  BKL   :  2099  (10.4%)
  DF    :   191  ( 0.9%)
  VASC  :   202  ( 1.0%)
  SCC   :   502  ( 2.5%)

VAL (5067 images):
  MEL   :   904  (17.8%)
  NV    :  2575  (50.8%)
  BCC   :   665  (13.1%)
  AK    :   173  ( 3.4%)
  BKL   :   525  (10.4%)
  DF    :    48  ( 0.9%)
  VASC  :    51  ( 1.0%)
  SCC   :   126  ( 2.5%)


In [48]:
# 10% subset TO REMOVE LATER
#train_df = train_df.sample(frac=0.01, random_state=Config["seed"]).reset_index(drop=True)
#val_df   = val_df.sample(frac=0.01,   random_state=Config["seed"]).reset_index(drop=True)
#test_df  = test_df.sample(frac=0.01,  random_state=Config["seed"]).reset_index(drop=True)
#print(f"Subset — train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

# Datasets
train_ds = ISICDataset(train_df, TRAIN_DIR, get_transforms("train"))
val_ds   = ISICDataset(val_df,   TRAIN_DIR, get_transforms("val"))
test_ds  = ISICDataset(test_df,  TEST_DIR,  get_transforms("test"))

# Dataloaders
train_loader = DataLoader(
    train_ds,
    batch_size=Config["batch_size"],
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=Config["batch_size"],
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=Config["batch_size"],
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [49]:
# compute class weights from standard ce so we can address class imbalance
counts = train_df["label"].value_counts().sort_index()
class_weights = (1.0 / counts).values
class_weights = class_weights / class_weights.sum() * len(class_weights)  # normalize
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

import torch
import torch.nn as nn
import torch.nn.functional as F

# Define Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, weight=None):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.weight = weight

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

# Build loss from Config
if Config["loss_fn"] == "weighted_ce":
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
elif Config["loss_fn"] == "ce":
    loss_fn = nn.CrossEntropyLoss()
elif Config["loss_fn"] == "focal":
    loss_fn = FocalLoss(
        alpha=Config.get("focal_alpha", 1),
        gamma=Config.get("focal_gamma", 2),
        weight=class_weights  # remove this if you don't want class weighting
    )
else:
    raise ValueError(f"Unknown loss_fn: {Config['loss_fn']}")

In [50]:
#Q: DID U RUN WANDB LOGIN IN TERMINAL?

In [51]:
#wandb.login(key="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")
#run the below for W&B login
#!/home/umw7eg/.local/bin/wandb login
#wandb.login(key="API_key")

os.environ["WANDB_NOTEBOOK_NAME"] = "EfficientNet_B0.ipynb"

wandb.init(
    entity="umw7eg_uva",
    project="ISIC2019",
    name=f"{Config['architecture']}_{Config['loss_fn']}_{Config['augmentation']}_pretrained{Config['pretrained']}",
    config={
        "architecture":  Config["architecture"],
        "loss_fn":       Config["loss_fn"],
        "augmentation":  Config["augmentation"],
        "pretrained":    Config["pretrained"],
        "freeze_bb":     Config["freeze_bb"],
        "img_size":      Config["img_size"],
        "batch_size":    Config["batch_size"],
        "epochs":        Config["epochs"],
        "lr":            Config["lr"],
        "val_split":     Config["val_split"],
        "patience":      Config["patience"],
        "seed":          Config["seed"],
    }
)

In [54]:
model     = build_model().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=Config["lr"])
scheduler = CosineAnnealingLR(optimizer, T_max=Config["epochs"], eta_min=1e-6)

best_bacc, best_epoch, no_improve = -1, 0, 0
history   = []
ckpt_path = os.path.join(CKPT_DIR, "efficientnet_b0_best.pt")

print(f"Training | pretrained={Config['pretrained']} | loss={Config['loss_fn']}\n")
t0 = time.time()

for epoch in range(1, Config["epochs"] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    (acc, bacc, sensitivity, specificity, 
    precision_per, recall_per, f1_per, 
    auc_per_class,
    cm, agg) = evaluate(model, val_loader, device)

    auc_macro = np.nanmean(list(auc_per_class.values()))
    
    log_dict = {
        "epoch":                 epoch,
        "train/loss":            train_loss,
        "val/acc":               acc,
        "val/bacc":              bacc,
        "val/sensitivity_macro": sensitivity.mean(),
        "val/specificity_macro": specificity.mean(),
        "val/macro_f1":          agg["macro_f1"],
        "val/wtd_f1":            agg["wtd_f1"],
        "val/macro_precision":   agg["macro_precision"],
        "val/macro_recall":      agg["macro_recall"],
        "val/auc_macro":         auc_macro,
    }
    for i, cls in enumerate(Config["classes"]):
        log_dict[f"val/sensitivity_{cls}"] = sensitivity[i]
        log_dict[f"val/specificity_{cls}"] = specificity[i]
        log_dict[f"val/f1_{cls}"]          = f1_per[i]
        log_dict[f"val/auc_{cls}"]         = auc_per_class[cls]   # per-class AUC

    wandb.log(log_dict)
    
    history.append({
        "epoch":      epoch,
        "train_loss": train_loss,
        
        "val_acc":    acc,
        "val_bacc":   bacc,
        
        "val_sens":   float(sensitivity.mean()),
        "val_spec":   float(specificity.mean()),
        
        "val_f1":     float(agg["macro_f1"]),
        "val_auc":    float(auc_macro),
    })

    print(f"Epoch {epoch:3d}/{Config['epochs']} | loss={train_loss:.4f} | "
          f"acc={acc:.4f} | bacc={bacc:.4f} | "
          f"macro_f1={agg['macro_f1']:.4f} | macro_auc={auc_macro:.4f}", end="")
    
    scheduler.step()

    if bacc > best_bacc:
        best_bacc, best_epoch = bacc, epoch
        no_improve = 0
        torch.save(model.state_dict(), ckpt_path)
        print("saved")
    else:
        no_improve += 1
        print(f"  (no improve {no_improve}/{Config['patience']})")
        if no_improve >= Config["patience"]:
            print(f"\nEarly stopping at epoch {epoch}.")
            break

print(f"\nDone in {(time.time()-t0)/60:.1f} min")

print(f"Best   | epoch={best_epoch} | bacc={best_bacc:.4f}")
print(f"Final  | acc={acc:.4f} | bacc={bacc:.4f} | "
      f"macro_f1={agg['macro_f1']:.4f} | macro_auc={auc_macro:.4f}")

history_path = os.path.join(CKPT_DIR, "history.json")
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"History saved to {history_path}")

Training | pretrained=True | loss=weighted_ce

Epoch   1/30 | loss=1.9225 | acc=0.5346 | bacc=0.5249 | macro_f1=0.3816 | macro_auc=0.8819saved
Epoch   2/30 | loss=1.2189 | acc=0.6175 | bacc=0.6122 | macro_f1=0.4959 | macro_auc=0.9176saved
Epoch   3/30 | loss=1.0235 | acc=0.6209 | bacc=0.6624 | macro_f1=0.5184 | macro_auc=0.9247saved
Epoch   4/30 | loss=0.8903 | acc=0.6511 | bacc=0.6877 | macro_f1=0.5455 | macro_auc=0.9344saved
Epoch   5/30 | loss=0.7481 | acc=0.6844 | bacc=0.6986 | macro_f1=0.6110 | macro_auc=0.9434saved
Epoch   6/30 | loss=0.6667 | acc=0.7053 | bacc=0.7085 | macro_f1=0.6205 | macro_auc=0.9478saved
Epoch   7/30 | loss=0.6008 | acc=0.7198 | bacc=0.7223 | macro_f1=0.6445 | macro_auc=0.9511saved
Epoch   8/30 | loss=0.5316 | acc=0.7207 | bacc=0.7328 | macro_f1=0.6553 | macro_auc=0.9529saved
Epoch   9/30 | loss=0.4523 | acc=0.7436 | bacc=0.7311 | macro_f1=0.6850 | macro_auc=0.9578  (no improve 1/10)
Epoch  10/30 | loss=0.4093 | acc=0.7383 | bacc=0.7316 | macro_f1=0.6616 | m

***!!IGNORE THE LINES BELOW FOR NOW***

In [ ]:
# ── Post-training report: best checkpoint on val set ─────────────────────────
model.load_state_dict(torch.load(ckpt_path, map_location=device))

(acc, bacc,
 sensitivity, specificity,
 precision_per, recall_per, f1_per,
 auc_per_class, cm, agg) = evaluate(model, val_loader, device)

auc_macro = np.nanmean(list(auc_per_class.values()))

# ── Scalar summary ────────────────────────────────────────────────────────────
print(f"{'='*54}")
print(f"  BACC          : {bacc:.4f}")
print(f"  Macro F1      : {agg['macro_f1']:.4f}")
print(f"  Weighted F1   : {agg['wtd_f1']:.4f}")
print(f"  Macro AUC     : {auc_macro:.4f}")
print(f"{'='*54}")
print(f"  {'Class':<6}  {'Sens':>6}  {'Spec':>6}  {'F1':>6}  {'AUC':>6}")
print(f"  {'-'*34}")
for i, cls in enumerate(Config["classes"]):
    print(f"  {cls:<6}  {sensitivity[i]:>6.4f}  {specificity[i]:>6.4f}  "
          f"{f1_per[i]:>6.4f}  {auc_per_class[cls]:>6.4f}")
print(f"{'='*54}")

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm_path = os.path.join(CKPT_DIR, "confusion_matrix.png")
plot_confusion_matrix(
    cm,
    class_names=Config["classes"],
    title=(f"Confusion Matrix — {Config['architecture']} "
           f"(loss={Config['loss_fn']}, aug={Config['augmentation']})\n"
           f"BACC={bacc:.4f}  |  Macro F1={agg['macro_f1']:.4f}  |  Macro AUC={auc_macro:.4f}"),
    save_path=cm_path,
)

# ── Log to W&B ────────────────────────────────────────────────────────────────
wandb.summary["best_val/bacc"]           = bacc
wandb.summary["best_val/macro_f1"]       = agg["macro_f1"]
wandb.summary["best_val/wtd_f1"]         = agg["wtd_f1"]
wandb.summary["best_val/macro_precision"]= agg["macro_precision"]
wandb.summary["best_val/macro_recall"]   = agg["macro_recall"]
wandb.summary["best_val/auc_macro"]      = auc_macro
for cls, auc in auc_per_class.items():
    wandb.summary[f"best_val/auc_{cls}"] = auc
wandb.log({"confusion_matrix": wandb.Image(cm_path)})
print("Metrics logged to W&B.")


In [55]:
def print_benchmark(model, label="Model"):
    params_total     = sum(p.numel() for p in model.parameters())
    params_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    size_mb          = params_total * 4 / 1024 ** 2  # float32

    # Inference speed
    model.eval()
    dummy = torch.randn(1, 3, 224, 224).to(device)
    with torch.no_grad():
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(50):
            model(dummy)
        end.record()
        torch.cuda.synchronize()
    inference_ms = start.elapsed_time(end) / 50

    # GFLOPs (requires thop)
    try:
        from thop import profile
        flops, _ = profile(model, inputs=(dummy,), verbose=False)
        gflops = flops / 1e9
    except:
        gflops = None

    bench = {
        "params_total":     params_total,
        "params_trainable": params_trainable,
        "size_mb":          round(size_mb, 2),
        "inference_ms":     round(inference_ms, 2),
        "gflops":           round(gflops, 4) if gflops else None
    }

    print(f"\n=== Benchmark: {label} ===")
    for k, v in bench.items():
        print(f"  {k}: {v}")

    return bench
        
model.load_state_dict(torch.load(ckpt_path, map_location=device))
bench = print_benchmark(model, label=f"EfficientNet-B0 ({Config['loss_fn']}, aug={Config['augmentation']})")
wandb.summary["benchmark/params_total"]     = bench["params_total"]
wandb.summary["benchmark/params_trainable"] = bench["params_trainable"]
wandb.summary["benchmark/size_mb"]          = bench["size_mb"]
wandb.summary["benchmark/inference_ms"]     = bench["inference_ms"]
wandb.summary["benchmark/gflops"]           = bench["gflops"]
wandb.summary["best_val_bacc"]              = best_bacc
wandb.summary["best_epoch"]                 = best_epoch
artifact = wandb.Artifact(
    name=f"efficientnet_b0_{Config['loss_fn']}_{Config['augmentation']}",
    type="model",
    description=f"Best checkpoint — val BACC {best_bacc:.4f} @ epoch {best_epoch}"
)
artifact.add_file(ckpt_path)
wandb.log_artifact(artifact)
print("Checkpoint artifact logged to W&B")
wandb.finish()
print("W&B run finished.")

# History saving
history_path = os.path.join(CKPT_DIR, "history.json")
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"History saved to {history_path}")


=== Benchmark: EfficientNet-B0 (weighted_ce, aug=standard) ===
  params_total: 4017796
  params_trainable: 4017796
  size_mb: 15.33
  inference_ms: 7.49
  gflops: None
Checkpoint artifact logged to W&B


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▅▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▃▃▄▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇█▇█████████
val/auc_AK,▁▄▄▆▆▇▇▇▇▇▇▇▇█▇▇▇█████████████
val/auc_BCC,▁▄▄▅▆▆▇▇▇▇▇▇▇▇████████████████
val/auc_BKL,▁▃▄▅▆▆▆▆▇▇▇▇▇▇▇▇██████████████
val/auc_DF,▁▆▇▅▇▇▇▇▇▇█▇▇█▄▅▆▆▆▆▇▇▆▆▆▆▆▆▆▆
val/auc_MEL,▁▃▃▄▅▆▆▆▆▇▇▇▇▇▇▇▇█████████████
val/auc_NV,▁▄▄▄▅▅▆▆▆▆▇▇▇▇▇▇▇█████████████
val/auc_SCC,▁▄▄▅▅▆▆▇▇▇▇▇▇▇█████▇▇█████████
+33,...


W&B run finished.
History saved to /scratch/umw7eg/isic2019/checkpoints/history.json
